# Assignment 8 and 9

## Decision Tree

For this lab, we are going to implement a decision tree based on the C4.5 algorithm. C4.5 provides several improvements over ID3 though the base structure is very similar. C4.5 removed the restriction that features must be categorical by dynamically defining a discrete attribute (based on numerical variables) that partitions the continuous attribute value into a discrete set of intervals. C4.5 converts the trained trees (i.e. the output of the ID3 algorithm) into sets of if-then rules.

We will start with our titanic dataset.

**Note:** Exercises can be autograded and count towards your lab and assignment score. Problems are graded for participation.

For developing this lab, we can a diabetes factors dataset. Description of the data is found https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset.

In [2]:
import pandas as pd
diabetes_df = pd.read_csv(
    f"https://raw.githubusercontent.com/Anderson-Lab/csc-466-student/refs/heads/main/data/diabetes_indicators.csv"
)
diabetes_df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


We need to do some simple preprocessing before our neural network can deal with this data.

In [3]:
features = ['Sex','Age','Education','Income','Fruits','Veggies','Smoker', "HighChol", "BMI"]
X = diabetes_df.loc[:,features][:1000]
X = X.dropna()
X

,Sex,Age,Education,Income,Fruits,Veggies,Smoker,HighChol,BMI
0,0.0,9.0,4.0,3.0,0.0,1.0,1.0,1.0,40.0
1,0.0,7.0,6.0,1.0,0.0,0.0,1.0,0.0,25.0
2,0.0,9.0,4.0,8.0,1.0,0.0,0.0,1.0,28.0
3,0.0,11.0,3.0,6.0,1.0,1.0,0.0,0.0,27.0
4,0.0,11.0,5.0,4.0,1.0,1.0,0.0,1.0,24.0
...,...,...,...,...,...,...,...,...,...
995,0.0,2.0,6.0,8.0,1.0,0.0,0.0,0.0,31.0
996,0.0,10.0,5.0,8.0,0.0,1.0,0.0,0.0,21.0
997,1.0,7.0,4.0,1.0,0.0,0.0,0.0,1.0,31.0
998,0.0,5.0,4.0,8.0,1.0,1.0,0.0,0.0,37.0


In [4]:
X.dtypes

,0
Sex,float64
Age,float64
Education,float64
Income,float64
Fruits,float64
Veggies,float64
Smoker,float64
HighChol,float64
BMI,float64


We will first implement ID3 before we move towards C4.5. This means we cannot handle continuois data such as ``BMI``. We will bin this into 20 categories. I picked 20 after trying a few different values. At this point, I do not know if it is a good selection or bad. This is part of the reason we will switch to C4.5.

In [5]:
X2 = X.copy()
X2['BMI'] = pd.cut(X2['BMI'],bins=20).astype(str) # bin Age up
X2['BMI'].value_counts()

,count
BMI,
"(28.9, 31.05]",194
"(26.75, 28.9]",147
"(24.6, 26.75]",141
"(22.45, 24.6]",125
"(31.05, 33.2]",88
"(20.3, 22.45]",65
"(33.2, 35.35]",64
"(35.35, 37.5]",41
"(37.5, 39.65]",27


In [6]:
t = diabetes_df.loc[X2.index,'Diabetes_012']
t

,Diabetes_012
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
995,0.0
996,0.0
997,0.0
998,0.0


**Function definitions below.** Open this by clicking mirror cell in tab or move around to more easily code.

In [107]:
import copy
import json

import numpy as np
import pandas as pd

from pandas.api.types import is_string_dtype
from pandas.api.types import is_numeric_dtype

def entropy(y):
    e = 0
    for i in y.unique():
      prob = sum(y == i) /len(y)
      e += -prob*np.log2(prob)
    return e

def gain(y,x):
    g = 0
    for i in x.unique():
      subset_y = y[x == i]
      weight = len(subset_y) / len(y)
      g += weight * entropy(subset_y)
    return entropy(y) - g

def gain_ratio(y,x):
    g = gain(y,x)
    return g/entropy(y)

def select_split(X,y):
    col = None
    gr = 0
    for c in X.columns:
      g = gain_ratio(y, X[c])
      if g > gr:
        gr, col = g, c
    return col,gr

def make_tree(X,y):

    if len(y.unique()) == 1:
        return y.unique()[0]

    if len(X.columns) == 0:
        return y.value_counts().idxmax()

    entropy_y = entropy(y)

    col, gr = select_split(X,y)
    if col is None:
        return y.value_counts().idxmax()

    tree = {col : {}}

    for val in X[col].dropna().unique():
        newX_val = X[X[col] == val].drop(columns=[col])
        tree[col][val] = make_tree(newX_val,y[X[col] == val])

    return tree

# if you want to print like me :)
def print_tree(tree):
    mytree = copy.deepcopy(tree)
    def fix_keys(tree):
        if type(tree) != dict:
            return tree #int(tree)
        new_tree = {}
        for key in list(tree.keys()):
            if type(key) == np.int64:
                new_tree[int(key)] = tree[key]
            else:
                new_tree[key] = tree[key]
        for key in new_tree.keys():
            new_tree[key] = fix_keys(new_tree[key])
        return new_tree
    mytree = fix_keys(mytree)
    print(json.dumps(mytree, indent=4, sort_keys=True))

def generate_rules(tree):
    rules = []
    #helper function to keep track of the current path on the tree
    def add_rules(node,path):
        #base case when you get to a leaf
        if not isinstance(node,dict):
            rules.append(path.copy() + [node])
            return

        if len(node) == 1:
          #go to next splitting feature
          feature = next(iter(node.keys()))
          children = node[feature]

          #extend rules for each branch
          for features, subtree in children.items():
            add_rules(subtree, path.copy() + [(feature, features)])
          return
        #if the node has multiple nodes then do the same thing
        for feature, children in node.items():
            if isinstance(children, dict):
                for feat_val, subtree in children.items():
                    add_rules(subtree, path.copy() + [(feature, feat_val)])
            else:
                add_rules(children, path.copy() + [(feature, None)])

    add_rules(tree, [])
    return rules

def split_col(x, y):
    x2 = list(x.copy())
    save_x = x.copy()
    x2.sort()
    splits = []
    gr = 0
    col = None
    for i in range(0, len(x2)-1):
        splits.append((x2[i] + x2[i+1]) / 2)
    for split in splits:
        x = x.apply(lambda x: True if x < split else False)
        g = gain_ratio(y, x)
        if g > gr:
            gr, col = g, split

        x = save_x.copy()
    return gr, col

def select_split2(X,y):
    newname, gr = None, -1
    #categorical
    for c in X.columns:
      #numerical
      if is_numeric_dtype(X[c]):
        g, col = split_col(X[c], y)
        if col is not None and g > gr:
          gr, newname = g, f"{c}<{col:.2f}"
      else:
        g = gain_ratio(y, X[c])
        if g > gr:
          gr, newname = g, c
    return newname ,gr

def make_tree2(X,y,min_split_count=5):

    if len(y.unique()) == 1:
        return y.unique()[0]

    num_elements = len(y)
    if num_elements < min_split_count:
        return y.value_counts().idxmax()

    if len(X.columns) == 0:
        return y.value_counts().idxmax()

    col, gr = select_split2(X,y)

    #print(col, gr)
    if col is None:
        return y.value_counts().idxmax()

    if "<" in col:
        col_name, val = col.split("<")
        val = float(val)
        if(X[col_name] < val).sum() == 0 or (X[col_name] >= val).sum() == 0:
            return y.value_counts().idxmax()

        return {col: {
            "True": make_tree2(X[X[col_name] < val].drop(columns=[col_name]), y[X[col_name] < val], min_split_count),
            "False":make_tree2(X[X[col_name] >= val].drop(columns=[col_name]), y[X[col_name] >= val], min_split_count)
        }}

    tree = {col : {}}

    for val in X[col].dropna().unique():
        newX_val = X[X[col] == val].drop(columns=[col])
        tree[col][val] = make_tree2(newX_val,y[X[col] == val], min_split_count)

    return tree

def evaluation(cm,positive_class=1):
    stats = {}
    ## BEGIN SOLUTION
    classes = list(cm.index)
    classes.remove(positive_class)
    tp = cm.loc[positive_class,positive_class]
    tn = fp = fn = 0
    for negative_class in classes:
        tn += cm.loc[negative_class,negative_class]
        fp += cm.loc[negative_class,positive_class]
        fn += cm.loc[positive_class,negative_class]
    for negative_class1 in classes:
        for negative_class2 in classes:
            if negative_class1 != negative_class2:
                tn += cm.loc[negative_class1,negative_class2]
    stats['accuracy'] = (tp+tn)/(tp+tn+fp+fn)
    stats['sensitivity/recall'] = tp/(tp+fn)
    stats['specificity'] = tn/(tn+fp)
    stats['precision'] = tp/(tp+fp)
    stats['F1'] = 2*stats['precision']*stats['sensitivity/recall']/(stats['precision']+stats['sensitivity/recall'])
    ## END SOLUTION
    return stats

def make_prediction(rules,x,default):
    for rule in rules:
      conditions = rule[:-1]
      prediction = rule[-1]
      matched = True
      for condition in conditions:
        feature, val = condition
        if "<" in feature:
          name, thr = feature.split("<")
          try:
            thr = float(thr)
          except Exception:
              matched = False
              break

          example_bool = False

          if name not in x.index:
              matched = False
              break

          example_bool = (x[name] < thr)

          # normalize val to boolean
          if isinstance(val, str):
              val_bool = val.lower() == "true"
          else:
            val_bool = bool(val)

          if example_bool != val_bool:
              matched = False
              break

        else:
          if feature not in x.index or x[feature] != val:
            matched = False
            break
      if matched:
        return prediction
    return (default)


#### Exercise 1
Construct a function called ``entropy`` that calculates the entropy of a set (Pandas Series Object)

In [54]:
e1 = entropy(t)
e2 = entropy(X2['Income'])
e1,e2

(np.float64(0.9013501922245392), np.float64(2.8994397663680536))

#### Exercise 2
Write a function called ``gain`` that calculates the information gain after splitting with a specific variable (Equation 12.2 from Marsland).

In [55]:
g1 = gain(t,X2['Sex'])
g2 = gain(t,X2['Income'])
g3 = gain(t,X2['Age'])
g1,g2,g3

(np.float64(0.00027367382888343617),
 np.float64(0.02399688322746074),
 np.float64(0.04855942918948519))

#### Exercise 3
C4.5 actually uses the gain ratio which is defined as the information gain "normalized" (divided) by the entropy before the split. You have written everything you need here. Just put it together.

In [56]:
gr1 = gain_ratio(t,X2['Sex'])
gr2 = gain_ratio(t,X2['Income'])
gr3 = gain_ratio(t,X2['Age'])
gr1,gr2,gr3

(np.float64(0.0003036265274521183),
 np.float64(0.026623263005287934),
 np.float64(0.053874098667067626))

#### Exercise 4
Define a function called ``select_split`` that chooses the column to place in the decision tree. This function returns the column name and the gain ratio for this column.

In [ ]:
col,gain_ratio_value = select_split(X2,t)
col,gain_ratio_value

('BMI', np.float64(0.06419179429194832))

#### Exercise 5
Now put it all together and construct a function called ``make_tree`` that returns a tree in the format shown below. This function is a recursive function. Think carefully about how to debug recursion (i.e., grab yourself a debugger such as https://docs.python.org/3/library/pdb.html). Think carefully the base cases.

In [ ]:
tree = make_tree(X2,t)
print_tree(tree)

{
    "BMI": {
        "(15.957, 18.15]": {
            "Sex": {
                "0.0": 0.0,
                "1.0": {
                    "Age": {
                        "9.0": 0.0,
                        "10.0": 2.0
                    }
                }
            }
        },
        "(18.15, 20.3]": {
            "Education": {
                "2.0": 0.0,
                "3.0": {
                    "Age": {
                        "8.0": 0.0,
                        "9.0": 0.0,
                        "10.0": 2.0,
                        "13.0": 2.0
                    }
                },
                "4.0": 0.0,
                "5.0": 0.0,
                "6.0": 0.0
            }
        },
        "(20.3, 22.45]": {
            "Age": {
                "1.0": 0.0,
                "2.0": 0.0,
                "3.0": 0.0,
                "4.0": 0.0,
                "5.0": 0.0,
                "6.0": 0.0,
                "7.0": 0.0,
                "8.0": 0.0,
              

#### Exercise 6
Create a recrusive function called ``generate_rules`` that returns an array of the rules from a tree. A rule is the form of:
```python
 [[('BMI', '(24.6, 26.75]'),
  ('Age', 7.0),
  ('Income', 8.0),
  ('Education', 5.0),
  2.0],...]
```
A single rule has a type of list. The last element in the list is the prediction, which is Survived=0 in this example. The tuples that preceed the last element are the conditions. Put another way, the above rule is equivalent to:
```python
if BMI == '(24.6, 26.75]' and Age == 7.0 and Income == 8.0 and Education == 5.0:
    predicted_value = 2.0
```

In [ ]:
rules = generate_rules(tree)
rules[:5] # the first 5 rules

[[('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(3.0)),
  np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(2.0)),
  np.float64(2.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(4.0)),
  np.float64(1.0)],
 [('BMI', '(39.65, 41.8]'), ('Age', np.float64(7.0)), np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(8.0)),
  ('Education', np.float64(4.0)),
  np.float64(2.0)]]

In [ ]:
rules

[[('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(3.0)),
  np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(2.0)),
  np.float64(2.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(9.0)),
  ('Income', np.float64(4.0)),
  np.float64(1.0)],
 [('BMI', '(39.65, 41.8]'), ('Age', np.float64(7.0)), np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(8.0)),
  ('Education', np.float64(4.0)),
  np.float64(2.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(8.0)),
  ('Education', np.float64(5.0)),
  ('Income', np.float64(1.0)),
  np.float64(2.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(8.0)),
  ('Education', np.float64(5.0)),
  ('Income', np.float64(7.0)),
  np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'),
  ('Age', np.float64(8.0)),
  ('Education', np.float64(2.0)),
  np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'), ('Age', np.float64(5.0)), np.float64(0.0)],
 [('BMI', '(39.65, 41.8]'), ('Age

#### Exercise 7
Create an improved function to create a tree called ``make_tree2``. This function is a recursive function. This function must add support for numeric columns, and it must incorporate a parameter that battles overfitting called ``min_split_count``. Minimum split count is incorporated as an additional base case. To implement, check to see if you have at least min_split_count items (i.e., num_elements >= min_split_count to split). The biggest change comes with the addition of numeric columns (Age and Fare in their original format). Please refer to the Marsland textbook for details on handling numeric values. In short, you try all possible locations to divide a numeric variable. For example, if your column has the values:
```
values = [1,3,2,5]
sorted_values = [1,2,3,5]
possible_splits = [<1.5,<2.5,<4]
```
Please make sure you denote your splits like I am doing above and how they are printed below.

In [70]:
# this takes a while to run for me on colab
tree2 = make_tree2(X,t,min_split_count=5)
print_tree(tree2)

{
    "Age<7.50": {
        "False": {
            "BMI<31.50": {
                "False": {
                    "Income<2.50": {
                        "False": {
                            "HighChol<0.50": {
                                "False": {
                                    "Veggies<0.50": {
                                        "False": {
                                            "Education<4.50": {
                                                "False": {
                                                    "Smoker<0.50": {
                                                        "False": {
                                                            "Sex<0.50": {
                                                                "False": {
                                                                    "Fruits<0.50": {
                                                                        "False": 2.0,
                                                             

#### Exercise 8
So how are we doing? We can put everything together and evaluate our solutions.

Create a function to make predictions called ``make_prediction``.

In [108]:
default = 0
from sklearn.model_selection import train_test_split

X2_train, X2_test, t_train, t_test = train_test_split(X2, t, test_size=0.3, random_state = 0)
X_train, X_test = X.loc[X2_train.index], X.loc[X2_test.index]

tree_id3 = make_tree(X2_train,t_train)
rules_id3 = generate_rules(tree_id3)
tree_c45 = make_tree2(X_train,t_train, min_split_count=20)
rules_c45 = generate_rules(tree_c45)

y_id3 = X2_test.apply(lambda x: make_prediction(rules_id3,x,default),axis=1)
y_c45 = X_test.apply(lambda x: make_prediction(rules_c45,x,default),axis=1)

In [109]:
def confusion_matrix(t,y,labels):
    cm = pd.DataFrame(columns=labels,index=labels)
    # actual is on the rows, pred on the columns
    for actual in cm.index:
        for pred in cm.columns:
            cm.loc[actual,pred] = sum( (t==actual) & (y==pred) )
    return cm

In [110]:
# Evaluate the id3
cm_id3 = confusion_matrix(t_test,y_id3,labels=[0,1,2])
print(cm_id3)
stats_id3 = evaluation(cm_id3,positive_class=2)
stats_id3

     0   1   2
0  145  40  28
1    5   4   1
2   42  13  22


{'accuracy': 0.72,
 'sensitivity/recall': 0.2857142857142857,
 'specificity': 0.8699551569506726,
 'precision': 0.43137254901960786,
 'F1': 0.34375}

In [111]:
# Evaluate the c45
cm_c45 = confusion_matrix(t_test,y_c45,labels=[0,1,2])
print(cm_c45)
stats_c45 = evaluation(cm_c45,positive_class=2)
stats_c45

     0  1   2
0  196  0  17
1   10  0   0
2   66  0  11


{'accuracy': 0.7233333333333334,
 'sensitivity/recall': 0.14285714285714285,
 'specificity': 0.9237668161434978,
 'precision': 0.39285714285714285,
 'F1': 0.2095238095238095}

In [98]:
cm_c45

,0,1,2
0,196,0,17
1,10,0,0
2,66,0,11


In [113]:
default = 0
from sklearn.model_selection import train_test_split

X2_train, X2_test, t_train, t_test = train_test_split(X2, t, test_size=0.3, random_state = 0)
X_train, X_test = X.loc[X2_train.index], X.loc[X2_test.index]

tree_c45_10 = make_tree2(X_train,t_train, min_split_count=10)
rules_c45_10 = generate_rules(tree_c45_10)
tree_c45_20 = make_tree2(X_train,t_train, min_split_count=20)
rules_c45_20 = generate_rules(tree_c45_20)
tree_c45_40 = make_tree2(X_train,t_train, min_split_count=40)
rules_c45_40 = generate_rules(tree_c45_40)
tree_c45_80 = make_tree2(X_train,t_train, min_split_count=80)
rules_c45_80 = generate_rules(tree_c45_80)

y_c45_10 = X_test.apply(lambda x: make_prediction(rules_c45_10,x,default),axis=1)

y_c45_20 = X_test.apply(lambda x: make_prediction(rules_c45_20,x,default),axis=1)

y_c45_40 = X_test.apply(lambda x: make_prediction(rules_c45_40,x,default),axis=1)

y_c45_80 = X_test.apply(lambda x: make_prediction(rules_c45_80,x,default),axis=1)

cm_c45_10 = confusion_matrix(t_test,y_c45_10,labels=[0,1,2])
stats_c45_10 = evaluation(cm_c45_10,positive_class=2)

cm_c45_20 = confusion_matrix(t_test,y_c45_20,labels=[0,1,2])
stats_c45_20 = evaluation(cm_c45_20,positive_class=2)

cm_c45_40 = confusion_matrix(t_test,y_c45_40,labels=[0,1,2])
stats_c45_40 = evaluation(cm_c45_40,positive_class=2)

# cm_c45_80 = confusion_matrix(t_test,y_c45_80,labels=[0,1,2])
# stats_c45_80 = evaluation(cm_c45_80,positive_class=2)

source = pd.DataFrame.from_records([stats_c45_10,stats_c45_20, stats_c45_40])
source



,accuracy,sensitivity/recall,specificity,precision,F1
0,0.726667,0.168831,0.919283,0.419355,0.240741
1,0.723333,0.142857,0.923767,0.392857,0.209524
2,0.720000,0.116883,0.928251,0.360000,0.176471


In [99]:
source = pd.DataFrame.from_records([stats_id3,stats_c45])
source['Method'] = ['ID3','C4.5']
source

,accuracy,sensitivity/recall,specificity,precision,F1,Method
0,0.720000,0.285714,0.869955,0.431373,0.343750,ID3
1,0.723333,0.142857,0.923767,0.392857,0.209524,C4.5


**Problem 1:** How do the two algorithms compare for this dataset?

**The algorithms have similar accuracy as they are both basically 72%. IDE has the better F1 making it the better model for this dataset. A high score reflects both high accuracy in positive predictions and comprehensive coverafe of acutal positive cases.**

**Problem 2:** Is this a robust experiment? How would you make it more robust? i.e., what are the flaws with what we did?

**This experiment is not robust for many reasons, but a few are that it does no cross validation with only training on a single train and test split. If we trained on more tests and training sets that would help our experiment.**

**Problem 3:** Repeat this experiment with min_split_count = 10, 20, 40, 80. How do the results change for C4.5?

**As I add more splits into the make tree, it made the model less and less accurate. The F1 score decrease because it is making the tree more conservative. This reduces vairance but inceases bias. I couldn't get results from the 80 minimum split count because the numbers were so low. **